# 02 SP - Pré-processamento: Perfil por Setor Censitário (SP)

Mesma lógica do notebook BA: agrega por `COD_SETOR`, calcula proporções de `COD_ESPECIE` e centróide.

Salva em `outputs/setores_features_sp.parquet`.

In [7]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pathlib import Path

con = duckdb.connect()
CENSO_SP = "'../data/censo 2022/35_SP.parquet'"
OUTPUT_DIR = Path('../outputs')

In [8]:
# Agrega por setor
df_setores = con.execute(f"""
    SELECT
        COD_SETOR,
        COUNT(*) AS total_enderecos,
        AVG(LATITUDE)  AS lat_centroide,
        AVG(LONGITUDE) AS lon_centroide,

        SUM(CASE WHEN COD_ESPECIE = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_domicilio_particular,
        SUM(CASE WHEN COD_ESPECIE = 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_domicilio_coletivo,
        SUM(CASE WHEN COD_ESPECIE = 3 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_estab_agropecuario,
        SUM(CASE WHEN COD_ESPECIE = 4 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_estab_ensino,
        SUM(CASE WHEN COD_ESPECIE = 5 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_estab_saude,
        SUM(CASE WHEN COD_ESPECIE = 6 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_estab_outras,
        SUM(CASE WHEN COD_ESPECIE = 7 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_construcao,
        SUM(CASE WHEN COD_ESPECIE = 8 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_estab_religioso,

        SUM(CASE WHEN COD_INDICADOR_FINALIDADE_CONST = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_finalidade_residencial,
        SUM(CASE WHEN COD_INDICADOR_FINALIDADE_CONST = 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_finalidade_comercial,
        SUM(CASE WHEN COD_INDICADOR_FINALIDADE_CONST = 3 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS prop_finalidade_mista

    FROM read_parquet({CENSO_SP})
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL
    GROUP BY COD_SETOR
    HAVING COUNT(*) >= 5
""").df()

print(f'Setores: {len(df_setores):,}')
df_setores.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setores: 98,902


,COD_SETOR,total_enderecos,lat_centroide,lon_centroide,prop_domicilio_particular,prop_domicilio_coletivo,prop_estab_agropecuario,prop_estab_ensino,prop_estab_saude,prop_estab_outras,prop_construcao,prop_estab_religioso,prop_finalidade_residencial,prop_finalidade_comercial,prop_finalidade_mista
0,355280905000699P,301,-23.629414,-46.793209,0.943522,0.0,0.000000,0.000000,0.000000,0.046512,0.009967,0.000000,0.006645,0.000000,0.0
1,351880010000708P,280,-23.443212,-46.411154,0.771429,0.0,0.000000,0.000000,0.003571,0.214286,0.010714,0.000000,0.007143,0.000000,0.0
2,352330510000042P,426,-24.288679,-47.036850,0.800469,0.0,0.000000,0.007042,0.007042,0.089202,0.091549,0.004695,0.091549,0.000000,0.0
3,355030887000160P,103,-23.486976,-46.463429,0.932039,0.0,0.000000,0.000000,0.000000,0.048544,0.009709,0.009709,0.000000,0.009709,0.0
4,350750605000212P,153,-22.846311,-48.430707,0.313725,0.0,0.013072,0.045752,0.000000,0.607843,0.013072,0.006536,0.000000,0.006536,0.0


In [9]:
print('Nulos por coluna:')
print(df_setores.isnull().sum())
df_setores = df_setores.dropna()
print(f'\nSetores após remover nulos: {len(df_setores):,}')

Nulos por coluna:
COD_SETOR                      0
total_enderecos                0
lat_centroide                  0
lon_centroide                  0
prop_domicilio_particular      0
prop_domicilio_coletivo        0
prop_estab_agropecuario        0
prop_estab_ensino              0
prop_estab_saude               0
prop_estab_outras              0
prop_construcao                0
prop_estab_religioso           0
prop_finalidade_residencial    0
prop_finalidade_comercial      0
prop_finalidade_mista          0
dtype: int64

Setores após remover nulos: 98,902


In [10]:
# Comparativo de diversidade: BA vs SP
# (proporção média de cada espécie)
FEATURES = [
    'prop_domicilio_particular', 'prop_domicilio_coletivo',
    'prop_estab_agropecuario', 'prop_estab_ensino', 'prop_estab_saude',
    'prop_estab_outras', 'prop_construcao', 'prop_estab_religioso',
    'prop_finalidade_residencial', 'prop_finalidade_comercial', 'prop_finalidade_mista'
]

medias_sp = df_setores[FEATURES].mean().rename('SP')

# carregar BA para comparar
import pyarrow.parquet as pq
df_ba_raw = pd.read_parquet(OUTPUT_DIR / 'setores_features.parquet')
# as features já estão normalizadas, recomputar médias originais não é possível direto
# mas podemos comparar os desvios padrões como proxy de diversidade

print('Desvio padrão SP (diversidade interna por feature):')
print(df_setores[FEATURES].std().round(4))

Desvio padrão SP (diversidade interna por feature):
prop_domicilio_particular      0.1433
prop_domicilio_coletivo        0.0180
prop_estab_agropecuario        0.0532
prop_estab_ensino              0.0105
prop_estab_saude               0.0112
prop_estab_outras              0.1169
prop_construcao                0.0557
prop_estab_religioso           0.0080
prop_finalidade_residencial    0.0510
prop_finalidade_comercial      0.0087
prop_finalidade_mista          0.0021
dtype: float64


In [11]:
# Normalização
scaler = StandardScaler()
X = scaler.fit_transform(df_setores[FEATURES])
print(f'Shape: {X.shape}')

Shape: (98902, 11)


In [12]:
# Salvar
df_out = df_setores[['COD_SETOR', 'total_enderecos', 'lat_centroide', 'lon_centroide']].copy()
df_out[FEATURES] = X

df_out.to_parquet(OUTPUT_DIR / 'setores_features_sp.parquet', index=False)
print(f'Salvo: outputs/setores_features_sp.parquet  ({len(df_out):,} setores)')

Salvo: outputs/setores_features_sp.parquet  (98,902 setores)
